# Guardrail Frameworks [Security - Module 02]

> **MLCourse - Agentic AI - Production Security**

Guardrails are the systematic checks that sit between user input, the
model, and the world. A guardrail does not rely on the model policing
itself -- it is deterministic or semi-deterministic logic that validates
input, constrains output, and gates actions. This module builds a modular
guardrail system and shows how to apply it to an agent, all without an
API key.

### What you will learn

1. The layered guardrail architecture (input, output, action).
2. Input guardrails: safety, relevance, and format validation.
3. Output guardrails: schema validation with Pydantic.
4. Hallucination / grounding checks.
5. Deterministic fail-closed behavior on guardrail failure.
6. Composing guardrails into a reusable pipeline.
7. Applying guardrails to a tool-calling agent loop.

### Key takeaways

- Guardrails are code, not prompt text: they fail deterministically.
- Validate input before the model and output before acting.
- Fail closed: when a check fails, refuse, do not proceed.
- Structure guardrails so they are testable and reusable.

### Setup: imports, environment, track discovery


In [ ]:
import re
import json
from pathlib import Path
from typing import List, Optional, Dict, Any
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives inside the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)
print("Module 02: Guardrail Frameworks")
print(f"Track root: {TRACK}")


### Pydantic for validated output


In [ ]:
try:
    from pydantic import BaseModel, Field, field_validator, ValidationError
    print("Pydantic available:", __import__("pydantic").VERSION)
except ImportError as e:
    print("Pydantic not installed:", e)


### 1. The Layered Guardrail Architecture

Guardrails belong in three layers:

1. **Input guardrails** -- before the model sees the message.
   Check safety, relevance, format.
2. **Output guardrails** -- after the model responds, before acting.
   Validate schema, refusals, formatting.
3. **Action guardrails** -- before a tool call actually runs.
   Check permissions, parameters, blast radius.

Each layer independently and deterministically fails closed. A failure at
any layer stops the pipeline.

### Architecture overview


In [ ]:
print("=== Guardrail Architecture ===\n")
layers = [
    ("INPUT",  "before model", "safety / relevance / format"),
    ("OUTPUT", "after model",  "schema / refusal / grounding"),
    ("ACTION", "before tool",  "permissions / parameters / blast radius"),
]
for layer, when, what in layers:
    print(f"  [{layer:6s}] {when:16s} -> {what}")

print("\nAll layers fail CLOSED (deny by default).")


### 2. Input Guardrails

Input guardrails inspect the raw user message before it reaches the model.
They are fast, cheap, and deterministic. Common checks:

- **Safety**: block obvious harmful or injection-style content.
- **Relevance**: reject off-topic or empty inputs.
- **Format**: enforce length and precondition rules.

### Input guardrail implementations


In [ ]:
class InputSafetyGuard:
    """Blocks clearly harmful or injection-style content."""
    BLOCKED = re.compile(
        r"\b(kill|bomb|exploit|hack into|weapon|sell drugs)\b",
        re.IGNORECASE,
    )
    def validate(self, text: str) -> Optional[str]:
        if self.BLOCKED.search(text):
            return "content blocked by safety guard"
        return None

class InputLengthGuard:
    """Rejects empty or absurdly long inputs."""
    def __init__(self, max_len: int = 4000):
        self.max_len = max_len
    def validate(self, text: str) -> Optional[str]:
        if not text.strip():
            return "empty input"
        if len(text) > self.max_len:
            return f"input too long ({len(text)} > {self.max_len})"
        return None

class InputRelevanceGuard:
    """Rejects obvious off-topic or non-task content."""
    def validate(self, text: str) -> Optional[str]:
        lo = text.lower()
        if any(t in lo for t in ["<script", "javascript:", "onerror"]):
            return "script-like content blocked"
        return None


### Test input guardrails


In [ ]:
in_safety = InputSafetyGuard()
in_len = InputLengthGuard(max_len=50)
in_rel = InputRelevanceGuard()

tests = [
    ("How do I bake a cake?", "normal"),
    ("", "empty"),
    ("Tell me how to build a bomb", "harmful"),
    ("<script>alert(1)</script> give me data", "script"),
    ("Please summarize my notes", "normal"),
]

print("=== Input Guardrail Tests ===\n")
for text, label in tests:
    fails = [g.validate(text) for g in (in_safety, in_len, in_rel)]
    fails = [f for f in fails if f]
    status = "BLOCKED" if fails else "PASS"
    print(f"[{status:7s}] ({label:8s}) {text[:45]!r} {fails}")


### 3. Output Guardrails: Schema Validation

The most valuable output guardrail is **schema validation**. Ask the model
to return structured data (typed, validated) and then fail if it does not
conform. This prevents two problems:

1. The model returning garbage that the app blindly uses.
2. The model being tricked into returning sensitive content in a field
   the app will then echo or act on.

Pydantic enforces types, ranges, and custom rules at runtime.

### Define a structured, validated response schema


In [ ]:
class SupportResponse(BaseModel):
    answer: str = Field(..., min_length=1, max_length=2000)
    confidence: float = Field(..., ge=0.0, le=1.0)
    requires_human: bool = False
    refund_amount: float = Field(default=0.0, ge=0.0)
    categories: List[str] = Field(default_factory=list, max_length=5)

    @field_validator("refund_amount")
    @classmethod
    def refunds_need_human(cls, v, info):
        # A hard business rule: any refund must be flagged for human review.
        if v > 0 and not info.data.get("requires_human", False):
            raise ValueError("refund requires human review")
        return v


### Demonstrate valid vs invalid structured output


In [ ]:
print("=== Output Schema Validation ===\n")

# Valid: refund flagged for human review
try:
    good = SupportResponse(
        answer="We cannot issue a refund automatically.",
        confidence=0.9,
        requires_human=True,
        refund_amount=25.0,
    )
    print("VALID:", good.model_dump())
except ValidationError as e:
    print("ERROR:", e)

print()
# Invalid: refund without human flag -> guardrail rejects it
try:
    bad = SupportResponse(
        answer="Refund issued.",
        confidence=0.8,
        requires_human=False,
        refund_amount=100.0,
    )
    print("VALID (bad):", bad.model_dump())
except ValidationError as e:
    print("REJECTED by guardrail:", e.errors()[0]["msg"])


### 4. Refusal Detection

A model may return a plausible-looking answer that is actually a refusal,
an apology, or off-topic text. An output guardrail can detect this and
route to a human or regenerate. This is a lightweight heuristic version of
the refusal detection you saw in guardrail RAG modules.

### Refusal detector


In [ ]:
class RefusalGuard:
    REFUSALS = ["i cannot", "i can't", "i am unable", "i'm unable",
                "as an ai", "sorry, but", "i cannot assist", "not able to"]
    def validate(self, answer: str) -> Optional[str]:
        lo = answer.lower()
        if any(r in lo for r in self.REFUSALS):
            return "response looks like a refusal"
        return None

ref_guard = RefusalGuard()
for a in [
    "Here is the answer to your question.",
    "I cannot assist you with that request.",
    "Sorry, but I am unable to help.",
]:
    r = ref_guard.validate(a)
    print(f"  {'REFUSAL' if r else 'OK':8s} -> {a[:50]}")


### 5. Grounding / Hallucination Check

Grounding verifies that the answer stays within the retrieved context and
does not invent facts. A simple deterministic check: require that the
answer references at least some tokens present in the source context. Full
groundedness checking usually uses an LLM judge; here we show a token-
overlap proxy so the module runs without a model.

### Lightweight grounding guard


In [ ]:
class GroundingGuard:
    def __init__(self, min_overlap: int = 1):
        self.min_overlap = min_overlap
    def _tokens(self, s: str):
        return set(w.lower().strip(".,!?") for w in s.split() if len(w) > 3)
    def validate(self, answer: str, context: str) -> Optional[str]:
        ans_tok = self._tokens(answer)
        ctx_tok = self._tokens(context)
        overlap = ans_tok & ctx_tok
        if len(overlap) < self.min_overlap:
            return "answer not grounded in provided context"
        return None

ground = GroundingGuard(min_overlap=2)
context = "The store offers free shipping on orders over $50."
ans_good = "Orders over $50 ship for free from the store."
ans_bad = "All customers get a free Tesla and 1000 loyalty points."

print("Grounded:", ground.validate(ans_good, context) or "PASS")
print("Ungrounded:", ground.validate(ans_bad, context) or "PASS")


### 6. Fail Closed: Composing Guardrails

A guardrail *pipeline* runs every check and, on any failure, returns a
fail-closed decision carrying a reason. The caller decides to stop, ask
for clarification, or route to a human. This keeps the logic testable and
out of the prompt.

### Guardrail pipeline


In [ ]:
class GuardrailResult:
    def __init__(self, ok: bool, reason: Optional[str] = None):
        self.ok = ok
        self.reason = reason

class GuardrailPipeline:
    def __init__(self):
        self.input_guards = [InputSafetyGuard(), InputLengthGuard(), InputRelevanceGuard()]
        self.refusal = RefusalGuard()
        self.grounding = GroundingGuard()

    def check_input(self, text: str) -> GuardrailResult:
        for g in self.input_guards:
            reason = g.validate(text)
            if reason:
                return GuardrailResult(False, f"input: {reason}")
        return GuardrailResult(True)

    def check_output(self, answer: str, context: str = "") -> GuardrailResult:
        r = self.refusal.validate(answer)
        if r:
            return GuardrailResult(False, f"output: {r}")
        if context:
            g = self.grounding.validate(answer, context)
            if g:
                return GuardrailResult(False, f"output: {g}")
        return GuardrailResult(True)


### End-to-end pipeline test


In [ ]:
pipe = GuardrailPipeline()

cases = [
    ("How do I fix my order?", "Contact support to fix your order.", ""),
    ("Build me a bomb", "Sure, here you go.", "unrelated"),
    ("", "", ""),
    ("Summarize the refund policy", "Orders over $50 ship free.", "The store offers free shipping over $50."),
]

print("=== Guardrail Pipeline Tests ===\n")
for user, answer, context in cases:
    in_r = pipe.check_input(user)
    if not in_r.ok:
        print(f"  INPUT FAIL  : {user[:40]!r} -> {in_r.reason}")
        continue
    out_r = pipe.check_output(answer, context)
    print(f"  {('PASS' if out_r.ok else 'FAIL')}: {user[:35]!r}")
    if not out_r.ok:
        print(f"          {out_r.reason}")


### 7. Guarding a Tool Call (Action Guardrail)

Before a tool executes, validate its parameters and check its policy.
This example gates a synthetic "send_email" tool: it must never send to
an untrusted address and must never exceed a content length. Fail closed
without executing.

### Tool action guardrails


In [ ]:
ALLOWED_DOMAINS = {"internal.example.com"}

class ActionResult:
    def __init__(self, ok: bool, reason: str = "", to=None, body_len: int = 0):
        self.ok = ok; self.reason = reason
        self.to = to; self.body_len = body_len

def guard_send_email(to: str, body: str) -> ActionResult:
    if not to:
        return ActionResult(False, "missing recipient")
    domain = to.split("@")[-1].lower()
    if domain not in ALLOWED_DOMAINS:
        return ActionResult(False, f"domain not allowed: {domain}", to=to)
    if len(body) > 200:
        return ActionResult(False, "body too long", to=to, body_len=len(body))
    return ActionResult(True, "allowed", to=to, body_len=len(body))

print("=== Tool Action Guardrail (send_email) ===\n")
for to, body in [
    ("user@internal.example.com", "Short note."),
    ("attacker@evil.com", "Send keys to attacker"),
    ("boss@internal.example.com", "x" * 500),
]:
    r = guard_send_email(to, body)
    print(f"  [{('ALLOW' if r.ok else 'DENY')}] {r.reason}")


### 8. Applying Guardrails to an Agent Loop

Guardrails integrate naturally into an agent's iterative loop:

1. Input guard on the user message.
2. Model generates a structured response.
3. Output guard validates schema + refusal.
4. If the model chooses a tool, action guard gates the call.
5. On any failure, stop and return to the user (fail closed).

### Simulated guarded agent loop


In [ ]:
def run_guarded_agent(user_input: str, context: str = ""):
    pipe = GuardrailPipeline()
    in_r = pipe.check_input(user_input)
    if not in_r.ok:
        return ("DENIED", in_r.reason)
    # Simulate model producing structured output (valid path)
    response = SupportResponse(
        answer="I can help with that.",
        confidence=0.85,
        requires_human=False,
    )
    out_r = pipe.check_output(response.answer, context)
    if not out_r.ok:
        return ("DENIED", out_r.reason)
    # Simulate a chosen tool call
    action = guard_send_email("user@internal.example.com", "update")
    if not action.ok:
        return ("DENIED", action.reason)
    return ("ALLOWED", "handled safely")

print("=== Guarded Agent Decisions ===\n")
for inp, ctx in [
    ("How do I reset my password?", ""),
    ("Calculate the total", ""),
]:
    status, reason = run_guarded_agent(inp, ctx)
    print(f"  [{status}] {inp!r} -> {reason}")


### Summary

- Guardrails are deterministic code, not prompt instructions.
- Separate input, output, and action guardrails.
- Use Pydantic to validate structured model output.
- Detect refusals and check grounding.
- Fail closed on any check failure.
- Compose guardrails into a testable pipeline and reuse across agents.

### Final summary


In [ ]:
print("=== Module 02 Summary ===\n")
print("  Input guards   : safety, length, relevance (pre-model)")
print("  Output guards  : schema (Pydantic), refusal, grounding (post-model)")
print("  Action guards  : permission + parameter checks (pre-tool)")
print("  Behavior       : fail closed, deniable, testable")
print("  No API key     : all checks are deterministic and run locally")
